In [19]:
%pip install pandas numpy openpyxl scipy matplotlib


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import pandas as pd
import numpy as np

print("Starting data load and cleaning...")

# --------------------------------------------------------
# 1. LOAD STATIC DATA & FILTER REGION
# --------------------------------------------------------
static_df = pd.read_excel('Data_2026/Static_2025.xlsx')
static_df['ISIN'] = static_df['ISIN'].astype(str).str.strip()

em_firms = static_df[static_df['Region'] == 'EM'].copy()
em_firms.dropna(subset=['ISIN'], inplace=True)
valid_em_isins = em_firms['ISIN'].unique().tolist()

print(f"-> Found {len(valid_em_isins)} valid EM firms in static data.")

# --------------------------------------------------------
# HELPER FUNCTION: LOAD E TRASPONI I DATI DATASTREAM
# --------------------------------------------------------
def load_and_transpose(filepath, is_monthly=False):
    # Carica il file Excel
    df = pd.read_excel(filepath)
    
    # Rimuovi righe di errore di Datastream se presenti
    df = df[~df['ISIN'].astype(str).str.contains('\$\$ER', na=False)]
    
    # Pulisci ISIN e impostali come indice (righe)
    df['ISIN'] = df['ISIN'].astype(str).str.strip()
    df.set_index('ISIN', inplace=True)
    
    # Rimuovi la colonna NAME perché non ci serve per i calcoli
    if 'NAME' in df.columns:
        df.drop(columns=['NAME'], inplace=True)
        
    # TRASPONI IL DATAFRAME: Le date diventano righe, gli ISIN diventano colonne
    df_t = df.T
    
    # Converti tutto in numeri (trasforma eventuali errori/testi in NaN)
    df_t = df_t.apply(pd.to_numeric, errors='coerce')
    
    # Se è un file mensile, formatta l'indice come data
    if is_monthly:
        df_t.index = pd.to_datetime(df_t.index)
    else:
        # Se è un file annuale (carbon), assicurati che gli anni siano numeri interi
        df_t.index = df_t.index.astype(int)
        
    return df_t

# --------------------------------------------------------
# 2. LOAD & CLEAN MONTHLY RETURN INDEX (RI)
# --------------------------------------------------------
ri_monthly = load_and_transpose('Data_2026/DS_RI_T_USD_M_2025.xlsx', is_monthly=True)

# Ora le colonne sono effettivamente gli ISIN!
em_columns_in_ri = [isin for isin in valid_em_isins if isin in ri_monthly.columns]
ri_em = ri_monthly[em_columns_in_ri].copy()

# Regola progetto: Prezzo < 0.5 diventa NaN
ri_em[ri_em < 0.5] = np.nan

# --------------------------------------------------------
# 3. CALCULATE SIMPLE RETURNS
# --------------------------------------------------------
returns_em = ri_em.pct_change()

print(f"-> Cleaned RI data and calculated simple returns for {len(returns_em.columns)} EM firms.")

# --------------------------------------------------------
# 4. LOAD SCOPE 1 & 2 EMISSIONS DATA
# --------------------------------------------------------
scope1_df = load_and_transpose('Data_2026/DS_CO2_SCOPE_1_Y_2025.xlsx', is_monthly=False)
scope2_df = load_and_transpose('Data_2026/DS_CO2_SCOPE_2_Y_2025.xlsx', is_monthly=False)

# Filtra per le stesse aziende EM
scope1_em = scope1_df[[isin for isin in valid_em_isins if isin in scope1_df.columns]].copy()
scope2_em = scope2_df[[isin for isin in valid_em_isins if isin in scope2_df.columns]].copy()

print("-> Scope 1 and Scope 2 data loaded and filtered.")
print("Data preparation complete!")

Starting data load and cleaning...
-> Found 702 valid EM firms in static data.
-> Cleaned RI data and calculated simple returns for 702 EM firms.
-> Scope 1 and Scope 2 data loaded and filtered.
Data preparation complete!


In [21]:
# --------------------------------------------------------
# CONTROLLO MANUALE: ANTEPRIMA ED ESPORTAZIONE (IN EXCEL)
# --------------------------------------------------------
# 1. Visualizza un'anteprima pulita nel Notebook (prime 5 date, prime 5 aziende)
print("--- Anteprima Rendimenti Mensili (Prime 5 righe/colonne) ---")
display(returns_em.iloc[:5, :5])

print("\n--- Anteprima Scope 1 (Prime 5 righe/colonne) ---")
display(scope1_em.iloc[:5, :5])

# 2. Esporta i dati puliti direttamente in file XLSX per aprirli nativamente
# Li salviamo nella tua cartella Data_2026 con il prefisso "CHECK_"
print("\nEsportazione in Excel in corso... (potrebbe volerci qualche secondo)")
returns_em.to_excel('Data_2026/CHECK_Returns_EM.xlsx')
scope1_em.to_excel('Data_2026/CHECK_Scope1_EM.xlsx')
scope2_em.to_excel('Data_2026/CHECK_Scope2_EM.xlsx')

print("✅ File di controllo esportati con successo! Vai nella cartella 'Data_2026' e aprili.")

--- Anteprima Rendimenti Mensili (Prime 5 righe/colonne) ---


ISIN,ARALUA010258,ARP125991090,ARSIDE010029,BMG211591018,BRABEVACNOR1
1999-12-31,NaN,NaN,NaN,NaN,NaN
2000-01-31,0.176511,-0.051861,-0.004877,NaN,-0.044175
2000-02-29,0.050002,0.178923,0.039543,NaN,-0.045539
2000-03-31,-0.023552,-0.105072,-0.002113,NaN,0.313941
2000-04-28,-0.016173,-0.161842,-0.071188,NaN,-0.001350



--- Anteprima Scope 1 (Prime 5 righe/colonne) ---


ISIN,ARALUA010258,ARP125991090,ARSIDE010029,BMG211591018,BRABEVACNOR1
1999,NaN,NaN,NaN,NaN,NaN
2000,NaN,NaN,NaN,NaN,NaN
2001,NaN,NaN,NaN,NaN,NaN
2002,NaN,NaN,NaN,NaN,NaN
2003,NaN,NaN,NaN,NaN,NaN



Esportazione in Excel in corso... (potrebbe volerci qualche secondo)
✅ File di controllo esportati con successo! Vai nella cartella 'Data_2026' e aprili.


In [22]:
# --------------------------------------------------------
# 5. DYNAMIC FILTERING: STALE PRICES & CARBON DATA
# --------------------------------------------------------
valid_investment_sets = {}

# We invest from the end of Y=2009 up to Y=2024 (for portfolio returns in 2010 to 2025)
years = range(2009, 2025)

for Y in years:
    # Define the 10-year estimation window: Dec of (Y-9) to Dec of Y
    start_date = f"{Y-9}-01-01"
    end_date = f"{Y}-12-31"
    
    # Slice the returns and prices for this 10-year window
    window_returns = returns_em.loc[start_date:end_date]
    window_prices = ri_em.loc[start_date:end_date]
    
    valid_isins_for_year = []
    
    for isin in returns_em.columns:
        # Rule 1: Price at the end of year Y must not be missing (we already set < 0.5 to NaN)
        # We check the last available month of year Y in the window
        if pd.isna(window_prices[isin].iloc[-1]):
            continue
            
        # Rule 2: Exclude if return is exactly 0 for strictly > 50% of the months in the window
        # Count how many months have exactly 0.0 return
        zero_returns_count = (window_returns[isin] == 0.0).sum()
        total_months = window_returns[isin].notna().sum()
        
        if total_months > 0 and (zero_returns_count / total_months) > 0.5:
            continue
            
        # Rule 3: Must have Scope 1 AND Scope 2 carbon data available at the end of year Y
        # Since carbon data is yearly, we check the index for year Y
        if Y in scope1_em.index and Y in scope2_em.index:
            if pd.isna(scope1_em.loc[Y, isin]) or pd.isna(scope2_em.loc[Y, isin]):
                continue
        else:
            continue # If the year is completely missing from carbon data
            
        # If it passes all rules, add it to the valid set!
        valid_isins_for_year.append(isin)
        
    valid_investment_sets[Y + 1] = valid_isins_for_year
    print(f"Year {Y+1}: {len(valid_isins_for_year)} valid EM firms.")

print("\nDynamic data cleaning complete! You now have the exact investment sets for each year.")

Year 2010: 65 valid EM firms.
Year 2011: 161 valid EM firms.
Year 2012: 188 valid EM firms.
Year 2013: 211 valid EM firms.
Year 2014: 221 valid EM firms.
Year 2015: 237 valid EM firms.
Year 2016: 261 valid EM firms.
Year 2017: 307 valid EM firms.
Year 2018: 345 valid EM firms.
Year 2019: 388 valid EM firms.
Year 2020: 445 valid EM firms.
Year 2021: 491 valid EM firms.
Year 2022: 529 valid EM firms.
Year 2023: 557 valid EM firms.
Year 2024: 571 valid EM firms.
Year 2025: 504 valid EM firms.

Dynamic data cleaning complete! You now have the exact investment sets for each year.
